# Assignment 3: Tiny Transformer — Tiny Shakespeare
Next-token prediction with a hand-built Transformer in PyTorch.

In [4]:
import os                                                                                                                      
for root, dirs, files in os.walk('/content/drive/MyDrive'):                                                                    
    for f in files:                                                                                                            
        print(os.path.join(root, f))

/content/drive/MyDrive/CS Resume 2023 (Projects).pdf
/content/drive/MyDrive/CS-D Resume.pdf
/content/drive/MyDrive/Raw_Summaries_Recent_studies_nf2573_columbia_edu_1706793613.docx
/content/drive/MyDrive/memory and the history.pdf
/content/drive/MyDrive/la capra history memory.pdf
/content/drive/MyDrive/Habermas-ConcerningPublicUse-1988.pdf
/content/drive/MyDrive/Raw_Summaries_Crenzel_Between_nf2573_columbia_edu_1708007963.docx
/content/drive/MyDrive/Bitcoin mining strategy.gdoc
/content/drive/MyDrive/Winslow outline.gdoc
/content/drive/MyDrive/Analysis of Epistle to Lady Margaret.gdoc
/content/drive/MyDrive/lab4.pdf
/content/drive/MyDrive/Winslow Good News Analysis.gdoc
/content/drive/MyDrive/Public sphere prompt.gdoc
/content/drive/MyDrive/Rigoberta Menchú.gslides
/content/drive/MyDrive/Rigoberta menchu notes.gdoc
/content/drive/MyDrive/Envisions.gdoc
/content/drive/MyDrive/Essay Draft PS.gdoc
/content/drive/MyDrive/Assignment 3 Outline.gdoc
/content/drive/MyDrive/Assignment 3 Litera

In [5]:
from google.colab import drive
drive.mount("/content/drive")

import shutil, os
os.makedirs("/content/outputs/experiments/baseline/checkpoints", exist_ok=True)
os.makedirs("/content/outputs/experiments/baseline/plots", exist_ok=True)

shutil.copy(
    "/content/drive/MyDrive/best.pt",
    "/content/outputs/experiments/baseline/checkpoints/best.pt"
)
shutil.copy(
    "/content/drive/MyDrive/loss_data.json",
    "/content/outputs/experiments/baseline/loss_data.json"
)
print("Baseline restored")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Baseline restored


## Phase 1 — Setup & Reproducibility

In [6]:
import math
import random
import sys
import urllib.request
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


In [7]:
# Environment detection — works identically locally and on Colab via the VS Code extension
ON_COLAB = "google.colab" in sys.modules

if ON_COLAB:
    import subprocess
    subprocess.run(["pip", "install", "tokenizers", "tqdm", "-q"], check=True)
    BASE_DIR = Path("/content")
    print("Running on Colab")
else:
    BASE_DIR = Path("..")
    print("Running locally")

DATA_DIR = BASE_DIR / "data"
CKPT_DIR = BASE_DIR / "outputs" / "checkpoints"
PLOT_DIR = BASE_DIR / "outputs" / "plots"

DATA_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR : {DATA_DIR.resolve()}")
print(f"CKPT_DIR : {CKPT_DIR.resolve()}")

Running on Colab
DATA_DIR : /content/data
CKPT_DIR : /content/outputs/checkpoints


In [8]:
# Central config — all hyperparameters live here
config = {
    "vocab_size":  500,
    "seq_len":      50,
    "d_model":     128,
    "n_heads":       4,
    "n_layers":      2,
    "ffn_dim":     256,
    "dropout":     0.1,
    "lr":         3e-4,
    "weight_decay": 0.01,
    "batch_size":   64,
    "num_epochs":   20,
}

## Phase 2 — Data Preparation & Tokenization

### 2a. Load the Tiny Shakespeare corpus

In [9]:
corpus_path = DATA_DIR / "input.txt"

if not corpus_path.exists():
    URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    print("Downloading Tiny Shakespeare...")
    urllib.request.urlretrieve(URL, corpus_path)

with open(corpus_path) as f:
    text = f.read()

print(f"Corpus length: {len(text):,} characters")
print("Sample:")
print(text[:300])

Corpus length: 1,115,394 characters
Sample:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


### 2b. BPE Tokenization

In [10]:
tokenizer_path = DATA_DIR / "tokenizer.json"

if not tokenizer_path.exists():
    print("Training BPE tokenizer...")
    tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(
        vocab_size=config["vocab_size"],
        special_tokens=["[UNK]", "[PAD]"],
    )
    tokenizer.train_from_iterator([text], trainer=trainer)
    tokenizer.save(str(tokenizer_path))
    print(f"Tokenizer saved to {tokenizer_path}")
else:
    tokenizer = Tokenizer.from_file(str(tokenizer_path))
    print(f"Tokenizer loaded from {tokenizer_path}")

actual_vocab_size = tokenizer.get_vocab_size()
print(f"Vocab size: {actual_vocab_size}")
config["vocab_size"] = actual_vocab_size

Training BPE tokenizer...
Tokenizer saved to /content/data/tokenizer.json
Vocab size: 500


### 2c. Encode corpus and create sliding-window sequences

In [11]:
encoding = tokenizer.encode(text)
token_ids = encoding.ids
print(f"Total tokens: {len(token_ids):,}")

SEQ_LEN = config["seq_len"]

inputs  = []
targets = []
for i in range(len(token_ids) - SEQ_LEN):
    inputs.append(token_ids[i : i + SEQ_LEN])
    targets.append(token_ids[i + 1 : i + SEQ_LEN + 1])

inputs  = torch.tensor(inputs,  dtype=torch.long)
targets = torch.tensor(targets, dtype=torch.long)
print(f"inputs shape:  {inputs.shape}")
print(f"targets shape: {targets.shape}")

Total tokens: 447,717
inputs shape:  torch.Size([447667, 50])
targets shape: torch.Size([447667, 50])


### 2d. Train / Val split (80 / 20)

In [12]:
n = len(inputs)
split = int(0.8 * n)

train_inputs,  val_inputs  = inputs[:split],  inputs[split:]
train_targets, val_targets = targets[:split], targets[split:]

print(f"Train sequences: {len(train_inputs):,}")
print(f"Val   sequences: {len(val_inputs):,}")

Train sequences: 358,133
Val   sequences: 89,534


In [13]:
class ShakespeareDataset(Dataset):
    def __init__(self, inputs, targets):
        self.inputs  = inputs
        self.targets = targets

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]


BATCH = config["batch_size"]

train_dataset = ShakespeareDataset(train_inputs, train_targets)
val_dataset   = ShakespeareDataset(val_inputs,   val_targets)

train_loader = DataLoader(train_dataset, batch_size=BATCH, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH, shuffle=False, drop_last=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")

Train batches: 5595
Val   batches: 1399


### 2e. Checkpoint — verify batch shapes

In [14]:
xb, yb = next(iter(train_loader))
print(f"Input  batch shape: {xb.shape}   (expected: [{BATCH}, {SEQ_LEN}])")
print(f"Target batch shape: {yb.shape}   (expected: [{BATCH}, {SEQ_LEN}])")

sample_tokens = xb[0].tolist()
decoded = tokenizer.decode(sample_tokens)
print(f"\nDecoded sample input:\n{decoded}")

Input  batch shape: torch.Size([64, 50])   (expected: [64, 50])
Target batch shape: torch.Size([64, 50])   (expected: [64, 50])

Decoded sample input:
an un li ck ' d be ar - w he l p That c ar ri es no im pr ess ion like the d am . And am I then a man to be be love d ? O m on str ous fa ul t , to


## Phase 3 — Model Implementation

### 3a. Sinusoidal Positional Encoding

In [15]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_seq_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_seq_len, d_model)
        pos = torch.arange(max_seq_len).unsqueeze(1).float()
        div = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)]

### 3b. RMSNorm

In [16]:
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-8):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return self.scale * x / rms

### 3c. Causal Multi-Head Self-Attention

In [17]:
class CausalMultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor):
        B, T, C = x.shape
        H, d_k = self.n_heads, self.d_k

        Q = self.q_proj(x).view(B, T, H, d_k).transpose(1, 2)
        K = self.k_proj(x).view(B, T, H, d_k).transpose(1, 2)
        V = self.v_proj(x).view(B, T, H, d_k).transpose(1, 2)

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)

        causal_mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        scores = scores.masked_fill(causal_mask, float("-inf"))

        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.attn_drop(attn_weights)

        out = (attn_weights @ V).transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out), attn_weights.detach()

### 3d. Feed-Forward Network

In [18]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, ffn_dim: int, dropout: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

### 3e. Transformer Block (pre-norm + residual)

In [19]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, ffn_dim: int, dropout: float = 0.0):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn  = CausalMultiHeadAttention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn   = FeedForward(d_model, ffn_dim, dropout)

    def forward(self, x: torch.Tensor):
        attn_out, attn_weights = self.attn(self.norm1(x))
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x, attn_weights

### 3f. Full Language Model

In [20]:
class TinyTransformer(nn.Module):
    def __init__(self, cfg: dict):
        super().__init__()
        V  = cfg["vocab_size"]
        D  = cfg["d_model"]
        H  = cfg["n_heads"]
        F  = cfg["ffn_dim"]
        L  = cfg["n_layers"]
        dr = cfg["dropout"]
        T  = cfg["seq_len"]

        self.token_emb = nn.Embedding(V, D)
        self.pos_enc   = SinusoidalPositionalEncoding(D, max_seq_len=T + 1)
        self.drop      = nn.Dropout(dr)
        self.blocks    = nn.ModuleList(
            [TransformerBlock(D, H, F, dr) for _ in range(L)]
        )
        self.norm_out  = RMSNorm(D)
        self.lm_head   = nn.Linear(D, V, bias=False)
        self.lm_head.weight = self.token_emb.weight

        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor):
        x = self.drop(self.pos_enc(self.token_emb(idx)))

        all_attn_weights = []
        for block in self.blocks:
            x, attn_weights = block(x)
            all_attn_weights.append(attn_weights)

        x = self.norm_out(x)
        logits = self.lm_head(x)
        return logits, all_attn_weights

### 3g. Sanity checks

In [21]:
model = TinyTransformer(config).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

dummy = torch.randint(0, config["vocab_size"], (4, config["seq_len"])).to(DEVICE)
with torch.no_grad():
    logits, attn_weights_list = model(dummy)

print(f"Logits shape:        {logits.shape}")
print(f"Attn weight layers:  {len(attn_weights_list)}")
print(f"NaNs in logits:      {logits.isnan().any().item()}")
print(f"Infs in logits:      {logits.isinf().any().item()}")

w = attn_weights_list[0]
T = config["seq_len"]
upper = torch.triu(w[0, 0], diagonal=1)
print(f"Max future attention weight: {upper.max().item():.2e}  (should be ~0)")

Total parameters: 327,552
Logits shape:        torch.Size([4, 50, 500])
Attn weight layers:  2
NaNs in logits:      False
Infs in logits:      False
Max future attention weight: 0.00e+00  (should be ~0)


## Phase 4 — Training Loop

### 4a. Loss, optimizer, and scheduler

In [22]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["lr"],
    weight_decay=config["weight_decay"],
)

total_steps = config["num_epochs"] * len(train_loader)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

print(f"Total training steps: {total_steps:,}")

Total training steps: 111,900


### 4b. Evaluation helper

In [23]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, n_batches = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits, _ = model(xb)
        loss = criterion(logits.view(-1, config["vocab_size"]), yb.view(-1))
        total_loss += loss.item()
        n_batches += 1
    return total_loss / n_batches

### 4c. Training loop

In [24]:
import json
baseline_ckpt = BASE_DIR / 'outputs' / 'experiments' / 'baseline' / 'checkpoints' / 'best.pt'
baseline_json = BASE_DIR / 'outputs' / 'experiments' / 'baseline' / 'loss_data.json'

if baseline_ckpt.exists() and baseline_json.exists():
    # Already trained via experiment runner — just load the results
    print('Baseline already trained — skipping Phase 4 training loop')
    with open(baseline_json) as f:
        _d = json.load(f)
    train_losses  = _d['train_losses']
    val_losses    = _d['val_losses']
    best_val_loss = min(val_losses)
else:
    train_losses = []
    val_losses   = []
    best_val_loss = float("inf")
    
    for epoch in range(config["num_epochs"]):
        model.train()
        epoch_loss, n_batches = 0.0, 0
    
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/{config['num_epochs']}", leave=False)
        for xb, yb in pbar:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
    
            optimizer.zero_grad()
            logits, _ = model(xb)
            loss = criterion(logits.view(-1, config["vocab_size"]), yb.view(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
    
            epoch_loss += loss.item()
            n_batches  += 1
            pbar.set_postfix(loss=f"{loss.item():.4f}")
    
        avg_train_loss = epoch_loss / n_batches
        avg_val_loss   = evaluate(model, val_loader)
        ppl            = math.exp(avg_val_loss)
    
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
    
        print(f"Epoch {epoch+1:02d} | train_loss={avg_train_loss:.4f} | val_loss={avg_val_loss:.4f} | PPL={ppl:.2f}")
    
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), CKPT_DIR / "best.pt")
            print(f"           ↳ New best checkpoint saved (val_loss={best_val_loss:.4f})")
    
    print(f"\nTraining complete. Best val PPL: {math.exp(best_val_loss):.2f}")


Baseline already trained — skipping Phase 4 training loop


### 4d. Final results summary

In [25]:
model.load_state_dict(torch.load(BASE_DIR / "outputs" / "experiments" / "baseline" / "checkpoints" / "best.pt", map_location=DEVICE))
final_val_loss = evaluate(model, val_loader)
final_ppl = math.exp(final_val_loss)

print(f"Final val loss : {final_val_loss:.4f}")
print(f"Final val PPL  : {final_ppl:.2f}")
print()
print("Epoch | train_loss | val_loss | PPL")
print("-" * 40)
for i, (tl, vl) in enumerate(zip(train_losses, val_losses), 1):
    print(f"  {i:02d}  |   {tl:.4f}   |  {vl:.4f}  | {math.exp(vl):.2f}")

Final val loss : 3.9632
Final val PPL  : 52.62

Epoch | train_loss | val_loss | PPL
----------------------------------------
  01  |   4.8039   |  4.4233  | 83.37
  02  |   4.0377   |  4.1980  | 66.55
  03  |   3.7793   |  4.1066  | 60.74
  04  |   3.6388   |  4.0521  | 57.52
  05  |   3.5535   |  4.0221  | 55.82
  06  |   3.4971   |  4.0041  | 54.82
  07  |   3.4576   |  3.9989  | 54.54
  08  |   3.4287   |  3.9961  | 54.38
  09  |   3.4062   |  3.9812  | 53.58
  10  |   3.3895   |  3.9727  | 53.13
  11  |   3.3760   |  3.9733  | 53.16
  12  |   3.3647   |  3.9693  | 52.95
  13  |   3.3563   |  3.9651  | 52.72
  14  |   3.3495   |  3.9664  | 52.80
  15  |   3.3441   |  3.9640  | 52.67
  16  |   3.3405   |  3.9648  | 52.71
  17  |   3.3378   |  3.9651  | 52.73
  18  |   3.3360   |  3.9632  | 52.62
  19  |   3.3348   |  3.9639  | 52.66
  20  |   3.3345   |  3.9638  | 52.66


In [26]:
import json                                                                                                                    
loss_data = {"train_losses": train_losses, "val_losses": val_losses}
with open("/content/drive/MyDrive/loss_data.json", "w") as f:                                                                  
    json.dump(loss_data, f)                                                                                                    
print("Loss data saved")

Loss data saved


In [27]:
print(f"Epochs completed: {len(train_losses)}")
print(f"Best val loss: {best_val_loss:.4f}")                                                                                   
print(f"Best val PPL: {math.exp(best_val_loss):.2f}")

Epochs completed: 20
Best val loss: 3.9632
Best val PPL: 52.62


## Phase 4d — Hyperparameter Experiments

In [28]:
# Experiments to run — each entry overrides keys from the base config
EXPERIMENTS = [
    {"name": "baseline",  "lr": 3e-4},
    {"name": "lr_high",   "lr": 1e-3},
    {"name": "lr_low",    "lr": 1e-4},
]

def experiment_dirs(name):
    exp_base = BASE_DIR / 'outputs' / 'experiments' / name
    ckpt_dir = exp_base / 'checkpoints'
    plot_dir = exp_base / 'plots'
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    plot_dir.mkdir(parents=True, exist_ok=True)
    return exp_base, ckpt_dir, plot_dir

In [29]:
def run_experiment(exp_cfg):
    name = exp_cfg['name']
    exp_base, ckpt_dir, plot_dir = experiment_dirs(name)

    # Skip if already trained
    if (ckpt_dir / 'best.pt').exists() and (exp_base / 'loss_data.json').exists():
        print(f'[{name}] Already trained — loading results')
        with open(exp_base / 'loss_data.json') as f:
            data = json.load(f)
        return data['train_losses'], data['val_losses']

    # Build config for this experiment
    exp_config = {**config, **{k: v for k, v in exp_cfg.items() if k != 'name'}}

    print(f'\n=== Experiment: {name} | lr={exp_config["lr"]} ===')
    torch.manual_seed(SEED)
    exp_model = TinyTransformer(exp_config).to(DEVICE)
    exp_criterion = nn.CrossEntropyLoss()
    exp_optimizer = torch.optim.AdamW(
        exp_model.parameters(), lr=exp_config['lr'], weight_decay=exp_config['weight_decay']
    )
    exp_total_steps = exp_config['num_epochs'] * len(train_loader)
    exp_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(exp_optimizer, T_max=exp_total_steps)

    exp_train_losses, exp_val_losses = [], []
    best_val = float('inf')

    for epoch in range(exp_config['num_epochs']):
        exp_model.train()
        epoch_loss, n_batches = 0.0, 0
        pbar = tqdm(train_loader, desc=f'  Epoch {epoch+1:02d}/{exp_config["num_epochs"]}', leave=False)
        for xb, yb in pbar:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            exp_optimizer.zero_grad()
            logits, _ = exp_model(xb)
            loss = exp_criterion(logits.view(-1, exp_config['vocab_size']), yb.view(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(exp_model.parameters(), 1.0)
            exp_optimizer.step()
            exp_scheduler.step()
            epoch_loss += loss.item(); n_batches += 1
            pbar.set_postfix(loss=f'{loss.item():.4f}')

        avg_train = epoch_loss / n_batches

        exp_model.eval()
        with torch.no_grad():
            total, nb_ = 0.0, 0
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits, _ = exp_model(xb)
                total += exp_criterion(logits.view(-1, exp_config['vocab_size']), yb.view(-1)).item()
                nb_ += 1
            avg_val = total / nb_

        exp_train_losses.append(avg_train)
        exp_val_losses.append(avg_val)
        print(f'  Epoch {epoch+1:02d} | train={avg_train:.4f} | val={avg_val:.4f} | PPL={math.exp(avg_val):.2f}')

        if avg_val < best_val:
            best_val = avg_val
            torch.save(exp_model.state_dict(), ckpt_dir / 'best.pt')

    # Save loss data
    with open(exp_base / 'loss_data.json', 'w') as f:
        json.dump({'train_losses': exp_train_losses, 'val_losses': exp_val_losses}, f)

    # Save per-experiment loss curve
    plt.figure(figsize=(8, 4))
    plt.plot(exp_train_losses, label='Train loss')
    plt.plot(exp_val_losses,   label='Val loss')
    plt.xlabel('Epoch'); plt.ylabel('Cross-entropy loss')
    plt.title(f'Loss curves — {name} (lr={exp_config["lr"]})')
    plt.legend(); plt.tight_layout()
    plt.savefig(plot_dir / 'loss_curves.png', dpi=150); plt.close()

    print(f'  Best val PPL: {math.exp(best_val):.2f} | saved to outputs/experiments/{name}/')
    return exp_train_losses, exp_val_losses


In [30]:
import json as _json
import matplotlib.pyplot as plt

all_results = {}
for exp_cfg in EXPERIMENTS:
    tl, vl = run_experiment(exp_cfg)
    all_results[exp_cfg['name']] = {'train': tl, 'val': vl, 'lr': exp_cfg['lr']}

[baseline] Already trained — loading results

=== Experiment: lr_high | lr=0.001 ===


  Epoch 01 | train=4.2619 | val=4.1119 | PPL=61.06


  Epoch 02 | train=3.6013 | val=4.0120 | PPL=55.26


  Epoch 03 | train=3.4681 | val=3.9918 | PPL=54.15


  Epoch 04 | train=3.3994 | val=3.9679 | PPL=52.88


...

  Epoch 16 | train=3.6069 | val=4.0535 | PPL=57.60


  Epoch 17 | train=3.6033 | val=4.0517 | PPL=57.49


  Epoch 18 | train=3.6009 | val=4.0512 | PPL=57.46


  Epoch 19 | train=3.5999 | val=4.0513 | PPL=57.47


  Epoch 20 | train=3.5997 | val=4.0511 | PPL=57.46
  Best val PPL: 57.46 | saved to outputs/experiments/lr_low/


### Comparison plot — all experiments

In [31]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for name, res in all_results.items():
    label = f"{name} (lr={res['lr']})"
    ax1.plot(res['train'], label=label)
    ax2.plot(res['val'],   label=label)

ax1.set_title('Train Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend()
ax2.set_title('Val Loss');   ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss'); ax2.legend()
plt.suptitle('Hyperparameter Comparison — Learning Rate', fontsize=13)
plt.tight_layout()

comparison_path = BASE_DIR / 'outputs' / 'plots' / 'comparison_lr.png'
plt.savefig(comparison_path, dpi=150); plt.close()
print(f'Saved comparison plot to {comparison_path}')

print('\nSummary:')
print(f'{"Experiment":<12} {"lr":<8} {"Best PPL":<10} {"Final PPL"}')
print('-' * 42)
for name, res in all_results.items():
    best_ppl  = math.exp(min(res['val']))
    final_ppl = math.exp(res['val'][-1])
    print(f'{name:<12} {str(res["lr"]):<8} {best_ppl:<10.2f} {final_ppl:.2f}')

Saved comparison plot to /content/outputs/plots/comparison_lr.png

Summary:
Experiment   lr       Best PPL   Final PPL
------------------------------------------
baseline     0.0003   52.62      52.66
lr_high      0.001    52.00      52.63
lr_low       0.0001   57.46      57.46


In [32]:
# Save all experiment outputs to Google Drive when on Colab
if ON_COLAB:
    import shutil
    shutil.copytree(
        str(BASE_DIR / 'outputs' / 'experiments'),
        '/content/drive/MyDrive/transformer_experiments',
        dirs_exist_ok=True,
    )
    shutil.copy(
        str(BASE_DIR / 'outputs' / 'plots' / 'comparison_lr.png'),
        '/content/drive/MyDrive/comparison_lr.png',
    )
    print('All experiment outputs saved to Google Drive')
else:
    print(f'Experiments saved locally at {(BASE_DIR / "outputs" / "experiments").resolve()}')

All experiment outputs saved to Google Drive


## Phase 5 — Visualization & Evaluation

### 5a. Loss curves

In [33]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

with open(BASE_DIR / 'outputs' / 'experiments' / 'baseline' / 'loss_data.json') as f:
    loss_data = json.load(f)
train_losses = loss_data['train_losses']
val_losses   = loss_data['val_losses']

epochs = range(1, len(train_losses) + 1)
plt.figure(figsize=(8, 4))
plt.plot(epochs, train_losses, label='Train loss')
plt.plot(epochs, val_losses,   label='Val loss')
plt.xlabel('Epoch')
plt.ylabel('Cross-entropy loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.tight_layout()
plt.savefig(BASE_DIR / 'outputs' / 'experiments' / 'baseline' / 'plots' / 'loss_curves.png', dpi=150)
plt.show()
print(f'Saved to {PLOT_DIR / "loss_curves.png"}')
print(f'Best val PPL: {math.exp(min(val_losses)):.2f} (epoch {val_losses.index(min(val_losses))+1})')

Saved to /content/outputs/plots/loss_curves.png
Best val PPL: 52.62 (epoch 18)


### 5b. Attention heatmaps

In [34]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Load best checkpoint into a fresh model
viz_model = TinyTransformer(config).to(DEVICE)
viz_model.load_state_dict(torch.load(BASE_DIR / 'outputs' / 'experiments' / 'baseline' / 'checkpoints' / 'best.pt', map_location=DEVICE))
viz_model.eval()

# Pick 3 samples from the val set
sample_indices = [0, 500, 1000]
samples = val_inputs[sample_indices].to(DEVICE)

with torch.no_grad():
    _, attn_weights_list = viz_model(samples)

for layer_idx, layer_weights in enumerate(attn_weights_list):
    # layer_weights: (B, H, T, T)
    for head_idx in range(config['n_heads']):
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle(f'Layer {layer_idx+1} — Head {head_idx+1}', fontsize=13)

        for ax, sample_idx, seq in zip(axes, sample_indices, samples):
            tokens = seq.tolist()
            labels = [tokenizer.decode([t]) for t in tokens]
            # Trim long labels
            labels = [l[:6] if len(l) > 6 else l for l in labels]

            w = layer_weights[sample_indices.index(sample_idx), head_idx].cpu().numpy()
            im = ax.imshow(w, cmap='viridis', aspect='auto', vmin=0, vmax=w.max())
            ax.set_title(f'Sample {sample_idx}', fontsize=10)
            ax.set_xlabel('Key position (attended to)')
            ax.set_ylabel('Query position (attending)')
            step = max(1, config['seq_len'] // 10)
            ticks = list(range(0, config['seq_len'], step))
            ax.set_xticks(ticks)
            ax.set_yticks(ticks)
            ax.set_xticklabels([labels[i] for i in ticks], rotation=90, fontsize=7)
            ax.set_yticklabels([labels[i] for i in ticks], fontsize=7)
            plt.colorbar(im, ax=ax, fraction=0.046)

        plt.tight_layout()
        fname = BASE_DIR / 'outputs' / 'experiments' / 'baseline' / 'plots' / f'attn_layer{layer_idx+1}_head{head_idx+1}.png'
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved {fname}')

Saved /content/outputs/experiments/baseline/plots/attn_layer1_head1.png
Saved /content/outputs/experiments/baseline/plots/attn_layer1_head2.png
Saved /content/outputs/experiments/baseline/plots/attn_layer1_head3.png
Saved /content/outputs/experiments/baseline/plots/attn_layer1_head4.png
Saved /content/outputs/experiments/baseline/plots/attn_layer2_head1.png
Saved /content/outputs/experiments/baseline/plots/attn_layer2_head2.png
Saved /content/outputs/experiments/baseline/plots/attn_layer2_head3.png
Saved /content/outputs/experiments/baseline/plots/attn_layer2_head4.png


### 5c. Perplexity summary

In [35]:
best_ppl  = math.exp(min(val_losses))
final_ppl = math.exp(val_losses[-1])
print(f'Best  val PPL (epoch {val_losses.index(min(val_losses))+1}): {best_ppl:.2f}')
print(f'Final val PPL (epoch {len(val_losses)}):                     {final_ppl:.2f}')

Best  val PPL (epoch 18): 52.62
Final val PPL (epoch 20):                     52.66


### 5d. Sample text generation

In [36]:
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=100, top_k=10):
    model.eval()
    ids = tokenizer.encode(prompt).ids
    ids = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(DEVICE)

    for _ in range(max_new_tokens):
        idx_cond = ids[:, -config['seq_len']:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :]  # last token

        # Top-k sampling
        if top_k is not None:
            topk_vals, _ = torch.topk(logits, top_k)
            logits[logits < topk_vals[:, -1:]] = float('-inf')

        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        ids = torch.cat([ids, next_id], dim=1)

    return tokenizer.decode(ids[0].tolist())


prompts = ['ROMEO:', 'To be or not to be']
for prompt in prompts:
    print(f'--- Prompt: "{prompt}" ---')
    print(generate(viz_model, tokenizer, prompt, max_new_tokens=80, top_k=10))
    print()

--- Prompt: "ROMEO:" ---
RO M E O : O ut your p ri ck s , my lord ; ' tis he ard s of you , And in this m at ter of the p ri de Of the wor ld ' s death , And I have se en for th to me . GLOUCESTER : A y , if this fe ast , I am g l as s , And yet , by my f o e , and f o llow it to

--- Prompt: "To be or not to be" ---
To be or not to be a l it t le d in thy life ? C on ce ed and my f al se : I will not be so much a man y man , I ' ll p ur po se , To see his c ous in s of s ad ly , And so on s of the f u ll of him ? D UCH ES S OF Y OR K : N ow , by my son , sir

